# 04 - Train/Test/Validation/Production Split

Splits engineered features into four datasets:

| Split | Source | Size | Purpose |
|-------|--------|------|---------|
| Train | 2019 data | 40% | Model training |
| Test | 2019 data | 10% | Model evaluation |
| Validation | 2019 data | 10% | Hyperparameter tuning |
| Production | 2020-2022 data | 100% sampled | Simulates real deployment |

2019 splits are stratified to maintain class ratio.
2020-2022 data is used as production to simulate the model being deployed on newer reviews.

**Prerequisite:** Run `03_feature_engineering.ipynb` first.

## 0. Install Dependencies

In [1]:
import importlib, subprocess

def install_if_missing(package):
    if importlib.util.find_spec(package) is None:
        print(f'Installing {package}...')
        subprocess.run(['pip', 'install', package, '--quiet'], check=True)
        print(f'{package} installed')
    else:
        print(f'{package} already installed, skipping')

install_if_missing('nltk')

import nltk
nltk.download('vader_lexicon', quiet=True)
print('Dependencies ready')

nltk already installed, skipping


Dependencies ready


## 1. Setup

In [2]:
import boto3, json, time
import pandas as pd
from sklearn.model_selection import train_test_split
from nltk.sentiment.vader import SentimentIntensityAnalyzer

with open('project_config.json') as f:
    cfg = json.load(f)

REGION           = cfg['REGION']
SOURCE_BUCKET    = cfg['SOURCE_BUCKET']
GLUE_DB          = cfg['GLUE_DB']
ATHENA_RESULTS   = cfg['ATHENA_RESULTS']
FEATURES_S3_PATH = cfg['FEATURES_S3_PATH']
FEATURE_COLS     = cfg['FEATURE_COLS']
TARGET_COL       = cfg['TARGET_COL']

session = boto3.Session(region_name=REGION)
s3      = session.client('s3')
athena  = session.client('athena')

RANDOM_STATE = 42  # fixed seed — everyone gets same splits

print(f'Region        : {REGION}')
print(f'Feature cols  : {FEATURE_COLS}')
print(f'Target col    : {TARGET_COL}')

Region        : us-east-1
Feature cols  : ['review_length', 'word_count', 'useful', 'funny', 'cool', 'vader_score']
Target col    : sentiment


## 2. Load 2019 Features from S3

Already engineered in notebook 03.

In [3]:
local_path = '/tmp/yelp_features.parquet'
s3.download_file(SOURCE_BUCKET, 'features/yelp_features.parquet', local_path)
df_2019 = pd.read_parquet(local_path)

print(f'2019 rows    : {len(df_2019):,}')
print(f'Columns      : {list(df_2019.columns)}')
print(f'Positive rate: {df_2019[TARGET_COL].mean()*100:.1f}%')

2019 rows    : 91,786
Columns      : ['review_id', 'review_length', 'word_count', 'useful', 'funny', 'cool', 'vader_score', 'sentiment']
Positive rate: 73.4%


## 3. Build Production Data from 2020-2022

Query, engineer features, and save — same pipeline as notebook 03.

In [4]:
def run_athena_query(sql):
    response = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={'Database': GLUE_DB},
        ResultConfiguration={'OutputLocation': ATHENA_RESULTS}
    )
    query_id = response['QueryExecutionId']
    while True:
        status = athena.get_query_execution(QueryExecutionId=query_id)
        state  = status['QueryExecution']['Status']['State']
        if state == 'SUCCEEDED':
            break
        elif state in ['FAILED', 'CANCELLED']:
            reason = status['QueryExecution']['Status']['StateChangeReason']
            raise Exception(f'Query {state}: {reason}')
        time.sleep(2)
    rows, next_token = [], None
    while True:
        kwargs = {'QueryExecutionId': query_id}
        if next_token:
            kwargs['NextToken'] = next_token
        page = athena.get_query_results(**kwargs)
        rows.extend(page['ResultSet']['Rows'])
        next_token = page.get('NextToken')
        if not next_token:
            break
    headers = [c['VarCharValue'] for c in rows[0]['Data']]
    data    = [[c.get('VarCharValue', '') for c in row['Data']] for row in rows[1:]]
    df = pd.DataFrame(data, columns=headers)
    # Fix dtypes — Athena returns everything as string
    df['stars']  = pd.to_numeric(df['stars'])
    df['useful'] = pd.to_numeric(df['useful'])
    df['funny']  = pd.to_numeric(df['funny'])
    df['cool']   = pd.to_numeric(df['cool'])
    df['date']   = pd.to_datetime(df['date'])
    return df

In [5]:
# Sample 100k from 2020-2022 to match 2019 scale
print('Loading 2020-2022 data from Athena...')
df_prod_raw = run_athena_query(
    'SELECT * FROM yelp_reviews_2020_2022 ORDER BY rand() LIMIT 100000'
)

print(f'Loaded {len(df_prod_raw):,} rows')
print(f'Star distribution:')
print(df_prod_raw['stars'].value_counts().sort_index())

Loading 2020-2022 data from Athena...


Loaded 100,000 rows
Star distribution:
stars
1    19977
2     6732
3     6791
4    13184
5    53316
Name: count, dtype: int64


In [6]:
# Apply same feature engineering as notebook 03
before = len(df_prod_raw)
df_prod_raw = df_prod_raw[df_prod_raw['stars'] != 3].copy()
df_prod_raw['sentiment']     = (df_prod_raw['stars'] >= 4).astype(int)
df_prod_raw['review_length'] = df_prod_raw['text'].str.len()
df_prod_raw['word_count']    = df_prod_raw['text'].str.split().str.len()

print(f'Dropped {before - len(df_prod_raw):,} three-star reviews')
print(f'Remaining: {len(df_prod_raw):,} rows')

# VADER scores
sid = SentimentIntensityAnalyzer()
print('Computing VADER scores...')
start = time.time()
df_prod_raw['vader_score'] = df_prod_raw['text'].apply(
    lambda x: sid.polarity_scores(x)['compound']
)
print(f'Done in {time.time() - start:.0f}s')

# Keep same columns as 2019 features
prod_data = df_prod_raw[[
    'review_id', 'review_length', 'word_count',
    'useful', 'funny', 'cool', 'vader_score', 'sentiment'
]].copy()

print(f'\nProduction positive rate: {prod_data[TARGET_COL].mean()*100:.1f}%')

Dropped 6,791 three-star reviews
Remaining: 93,209 rows
Computing VADER scores...


Done in 68s

Production positive rate: 71.3%


## 4. Split 2019 Data into Train/Test/Validation

Stratified to maintain ~73% positive rate across all splits.

In [7]:
# Step 1: split off 40% for train, 60% remaining
train_data, temp_data = train_test_split(
    df_2019,
    test_size=0.333,
    random_state=RANDOM_STATE,
    stratify=df_2019[TARGET_COL]
)

# Step 2: split remaining 20% into test (10%) and val (10%)
test_data, val_data = train_test_split(
    temp_data,
    test_size=0.5,
    random_state=RANDOM_STATE,
    stratify=temp_data[TARGET_COL]
)

total = len(df_2019)
print('Split results:')
print(f'  Train      : {len(train_data):>6,} rows ({len(train_data)/total*100:.1f}%)')
print(f'  Test       : {len(test_data):>6,} rows ({len(test_data)/total*100:.1f}%)')
print(f'  Validation : {len(val_data):>6,} rows ({len(val_data)/total*100:.1f}%)')
print(f'  Production : {len(prod_data):>6,} rows (2020-2022)')

Split results:
  Train      : 61,221 rows (66.7%)
  Test       : 15,282 rows (16.6%)
  Validation : 15,283 rows (16.7%)
  Production : 93,209 rows (2020-2022)


## 5. Verify Class Ratio Preserved

In [8]:
print('Positive rate per split:')
for name, split in [('Train', train_data), ('Test', test_data),
                    ('Validation', val_data), ('Production', prod_data)]:
    rate = split[TARGET_COL].mean() * 100
    print(f'  {name:<12}: {rate:.1f}%')

Positive rate per split:
  Train       : 73.4%
  Test        : 73.4%
  Validation  : 73.4%
  Production  : 71.3%


## 6. Save All Splits to S3

In [9]:
splits = {
    'train':      train_data,
    'test':       test_data,
    'validation': val_data,
    'production': prod_data,
}

split_paths = {}

for split_name, split_df in splits.items():
    local_path = f'/tmp/{split_name}.parquet'
    s3_key     = f'splits/{split_name}.parquet'

    split_df.to_parquet(local_path, index=False)
    s3.upload_file(local_path, SOURCE_BUCKET, s3_key)

    split_paths[split_name] = f's3://{SOURCE_BUCKET}/{s3_key}'
    print(f'{split_name:<12}: saved to s3://{SOURCE_BUCKET}/{s3_key}')

print('\nAll splits saved')

train       : saved to s3://aai540-group1-yelp-data/splits/train.parquet


test        : saved to s3://aai540-group1-yelp-data/splits/test.parquet
validation  : saved to s3://aai540-group1-yelp-data/splits/validation.parquet


production  : saved to s3://aai540-group1-yelp-data/splits/production.parquet

All splits saved


## 7. Update Config for Modeling Notebook

In [10]:
cfg['SPLIT_PATHS']  = split_paths
cfg['RANDOM_STATE'] = RANDOM_STATE

with open('project_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print('Config updated')
print(json.dumps(cfg, indent=2))

Config updated
{
  "REGION": "us-east-1",
  "SOURCE_BUCKET": "aai540-group1-yelp-data",
  "GLUE_DB": "yelp_reviews_db",
  "ATHENA_RESULTS": "s3://aai540-group1-yelp-data/athena-results/qmou/",
  "TABLES": [
    "yelp_reviews_2019",
    "yelp_reviews_2020_2022"
  ],
  "FEATURE_GROUP_NAME": "yelp-review-features",
  "FEATURES_S3_PATH": "s3://aai540-group1-yelp-data/features/yelp_features.parquet",
  "FEATURE_COLS": [
    "review_length",
    "word_count",
    "useful",
    "funny",
    "cool",
    "vader_score"
  ],
  "TARGET_COL": "sentiment",
  "SPLIT_PATHS": {
    "train": "s3://aai540-group1-yelp-data/splits/train.parquet",
    "test": "s3://aai540-group1-yelp-data/splits/test.parquet",
    "validation": "s3://aai540-group1-yelp-data/splits/validation.parquet",
    "production": "s3://aai540-group1-yelp-data/splits/production.parquet"
  },
  "RANDOM_STATE": 42
}
